# Importações

In [1]:
import abc

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import warnings

from matplotlib.ticker import FuncFormatter
from shap.plots import colors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

from COA import COA
from pyESN import ESN

from pyswarms.single import GlobalBestPSO
from sklearn.model_selection import TimeSeriesSplit, train_test_split

from anneal import Annealer
from wsb import WSB

MODELOS = ["ESN", "MLP", "RF", "XGBoost"]
MODELOS_WSB = ["WSB-GLOBAL", "WSB-LOCAL"]
SEEDS = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
OTIMIZADORES = ["PSO", "SA"]
N_ITER = 15
N_SOLUCOES = 15
K_FOLDS = 5

warnings.filterwarnings('ignore')


def reset_seed(rnd_seed):
    os.environ['PYTHONHASHSEED'] = '0'
    random.seed(rnd_seed)
    np.random.seed(rnd_seed)


def calcular_rrmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)

    mean_y_true = np.mean(y_true)

    rrmse = rmse / mean_y_true
    return rrmse


reset_seed(100)


C:\Users\eduar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração dos Otimizadores
## Otimizador Base

In [2]:
class Otimizador:

    @abc.abstractmethod
    def __init__(self, dataset, features, n_solucoes=10, n_iteracoes=10, seed=1000):
        reset_seed(seed)
        self.nome_modelo = None
        self.dataset = dataset
        self.features = features
        self.seed = seed
        self.n_solucoes = n_solucoes
        self.n_iteracoes = n_iteracoes
        self.solucoes = []
        self.iteracoes = []

    @abc.abstractmethod
    def run(self):
        pass

    @abc.abstractmethod
    def run_objective_function(self, parametros):
        pass

    def objective_function(self, modelo):
        cvs_todos = []

        for campus, dados in self.dataset.groupby("CAMPUS"):
            previsoes = []
            valores_reais = []

            for i_treino, i_teste in TimeSeriesSplit(n_splits=K_FOLDS, test_size=1).split(dados):
                x = dados[self.features]
                y = dados["CONSUMO"]

                x_treino = x.iloc[i_treino]
                y_treino = y.iloc[i_treino]
                x_teste = x.iloc[i_teste]
                y_teste = y.iloc[i_teste]

                modelo.fit(x_treino, y_treino)

                y_previsto = modelo.predict(x_teste)[0]
                previsoes.append(y_previsto)
                valores_reais.append(y_teste)

            cvs_todos.append(calcular_rrmse(valores_reais, previsoes).mean())

        return np.array(cvs_todos).mean()

    def iteracoes_dataframe(self):
        df = pd.DataFrame()
        for i in range(len(self.iteracoes)):
            part = self.iteracoes[i]
            df = pd.concat([df, pd.DataFrame.from_dict(part.to_dict(), orient='index').T], ignore_index=True)
        return df

    def salvar_csv(self, subpath=""):
        pd_df = self.iteracoes_dataframe()
        pd_df.to_csv(f"resultados/otimização - regressão/{subpath}{self.nome_modelo} SEED {self.seed}.csv", sep=";", decimal=".",
                     index=True)


### ESN

In [3]:
class SolucaoESN:
    def __init__(self):
        self.fitness = None
        self.n_reservoirs = 0
        self.sparsity = 0
        self.spectral_radius = 0

    def to_dict(self):
        return {
            "Reservoirs": self.n_reservoirs,
            "Sparsity": self.sparsity,
            "Spectral Radius": self.spectral_radius,
            "Fitness": self.fitness,
        }

### MLP

In [4]:
class SolucaoMLP:
    def __init_(self):
        self.fitness = None
        self.hidden_layer_sizes = 0
        self.alpha = 0
        self.activation = None

    def to_dict(self):
        return {
            "Hidden Layers": self.hidden_layer_sizes,
            "Alpha": self.alpha,
            "Activation": self.activation,
            "Fitness": self.fitness,
        }


### Random Forest

In [5]:
class SolucaoRF:
    def __init_(self):
        self.fitness = None
        self.estimators = 0
        self.max_depth = 0
        self.min_samples_split = 0
        self.min_samples_leaf = 0

    def to_dict(self):
        return {
            "N_estimators": self.estimators,
            "Max_depth": self.max_depth,
            "Min_samples_split": self.min_samples_split,
            "Min_samples_leaf": self.min_samples_leaf,
            "Fitness": self.fitness,
        }

### XGBoost

In [6]:
class SolucaoXGB:
    def __init_(self):
        self.fitness = None
        self.estimators = 0
        self.max_depth = 0
        self.booster = None
        self.reg_lambda = 0
        self.reg_alpha = 0

    def to_dict(self):
        return {
            "N_estimators": self.estimators,
            "Max_depth": self.max_depth,
            "Booster": self.booster,
            "Lambda": self.reg_lambda,
            "Alpha": self.reg_alpha,
            "Fitness": self.fitness,
        }


### WSB

In [7]:
class SolucaoWSB:
    def __init__(self):
        self.fitness = None
        self.strong_predictor = None
        self.weight_g = 0

    def to_dict(self):
        return {
            "Strong_predictor": self.strong_predictor,
            "Weight_g": self.weight_g,
            "Fitness": self.fitness,
        }

## Simmulated Annealing (SA)

In [8]:
class SimulatedAnneal(Annealer):
    def __init__(self, params, objective_function, max_iter):
        self.params = params
        self.objective_function = objective_function
        super().__init__(self.random_initial_state())
        self.steps = max_iter
        self.Tmin = 0.0001
        self.Tmax = 1
        self.updates = 0
        self.anneal()

    def random_initial_state(self):
        initial_state = {}
        for key in self.params.keys():
            initial_state[key] = random.choice(self.params[key])
        return initial_state

    def move(self):
        atual = self.state
        for key in self.params.keys():
            valor = atual[key]
            opcoes = len(self.params[key])
            if opcoes <= 2:
                self.state[key] = random.choice(self.params[key])
            else:
                intervalo = int(np.round(self.T * opcoes))
                diferencas = np.abs(np.array(self.params[key]) - valor)
                index = np.argmin(diferencas)
                inicio = max(0, index - intervalo)
                fim = min(opcoes - 1, index + intervalo)
                if inicio == fim:
                    self.state[key] = self.params[key][inicio]
                else:
                    self.state[key] = self.params[key][random.choice(range(inicio, fim))]

    def energy(self):
        return self.objective_function(self.state)

### ESN

In [9]:
class SAESN(Otimizador):

    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "SA-ESN"
        self.run()

    def run(self):
        parametros = {
            "n_reservoirs": range(10, 400),
            "sparsity": np.arange(0.001, 0.8, 0.001),
            "spectral_radius": np.arange(0.001, 0.8, 0.001),
        }
        SimulatedAnneal(parametros, self.run_objective_function, max_iter=self.n_iteracoes * self.n_solucoes)

    def run_objective_function(self, parametros):
        solucao = SolucaoESN()
        solucao.n_reservoirs = int(parametros["n_reservoirs"])
        solucao.sparsity = np.round(parametros["sparsity"], 4)
        solucao.spectral_radius = np.round(parametros["spectral_radius"], 4)

        search = list(filter(lambda par:
                             par.n_reservoirs == solucao.n_reservoirs and
                             par.sparsity == solucao.sparsity and
                             par.spectral_radius == solucao.spectral_radius, self.solucoes))

        if search:
            solucao = search[0]

        else:
            modelo = ESN(n_inputs=self.dataset[self.features].shape[1],
                         n_outputs=1,
                         n_reservoir=solucao.n_reservoirs,
                         sparsity=solucao.sparsity,
                         spectral_radius=solucao.spectral_radius,
                         random_state=self.seed)

            solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return solucao.fitness

### MLP

In [10]:
class SAMLP(Otimizador):
    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "SA-MLP"
        self.ACTIVATIONS = ["identity", "logistic", "tanh", "relu"]
        self.run()

    def run(self):
        parametros = {
            "hidden_layer_sizes": range(10, 400),
            "alpha": np.arange(0, 1, 0.01),
            "activation": range(0, 4),
        }
        SimulatedAnneal(parametros, self.run_objective_function, max_iter=self.n_iteracoes * self.n_solucoes)

    def run_objective_function(self, parametros):
        solucao = SolucaoMLP()
        solucao.hidden_layer_sizes = int(parametros["hidden_layer_sizes"])
        solucao.alpha = round(parametros["alpha"], 4)
        solucao.activation = self.ACTIVATIONS[int(parametros["activation"])]

        search = list(filter(lambda par:
                             par.hidden_layer_sizes == solucao.hidden_layer_sizes and
                             par.alpha == solucao.alpha and
                             par.activation == solucao.activation, self.solucoes))

        if search:
            solucao = search[0]

        else:
            modelo = MLPRegressor(hidden_layer_sizes=(solucao.hidden_layer_sizes,),
                                  activation=solucao.activation,
                                  alpha=solucao.alpha,
                                  random_state=self.seed)

            solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return solucao.fitness

### Random Forest


In [11]:
class SARF(Otimizador):
    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "SA-RF"
        self.run()

    def run(self):
        parametros = {
            "estimators": range(10, 400),
            "max_depth": range(10, 400),
            "min_samples_split": range(2, 50),
            "min_samples_leaf": range(2, 50),
        }
        SimulatedAnneal(parametros, self.run_objective_function, max_iter=self.n_iteracoes * self.n_solucoes)

    def run_objective_function(self, parametros):
        solucao = SolucaoRF()
        solucao.estimators = int(parametros["estimators"])
        solucao.max_depth = int(parametros["max_depth"])
        solucao.min_samples_split = int(parametros["min_samples_split"])
        solucao.min_samples_leaf = int(parametros["min_samples_leaf"])

        search = list(filter(lambda par:
                             par.estimators == solucao.estimators and
                             par.max_depth == solucao.max_depth and
                             par.min_samples_split == solucao.min_samples_split and
                             par.min_samples_leaf == solucao.min_samples_leaf, self.solucoes))

        if search:
            solucao = search[0]

        else:
            modelo = RandomForestRegressor(random_state=self.seed,
                                           n_estimators=solucao.estimators,
                                           max_depth=solucao.max_depth,
                                           min_samples_split=solucao.min_samples_split,
                                           min_samples_leaf=solucao.min_samples_leaf)

            solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return solucao.fitness


### XGBoost

In [12]:
class SAXGB(Otimizador):
    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "SA-XGBoost"
        self.BOOSTERS = ["gbtree", "gblinear", "dart"]
        self.run()

    def run(self):
        parametros = {
            "estimators": range(10, 400),
            "max_depth": range(10, 400),
            "booster": range(0, 2),
            "reg_lambda": np.arange(0, 1, 0.005),
            "reg_alpha": np.arange(0, 1, 0.005),
        }
        SimulatedAnneal(parametros, self.run_objective_function, max_iter=self.n_iteracoes * self.n_solucoes)

    def run_objective_function(self, parametros):
        solucao = SolucaoXGB()
        solucao.estimators = int(parametros["estimators"])
        solucao.max_depth = int(parametros["max_depth"])
        solucao.booster = self.BOOSTERS[int(parametros["booster"])]
        solucao.reg_lambda = float(parametros["reg_lambda"])
        solucao.reg_alpha = float(parametros["reg_alpha"])

        search = list(filter(lambda par:
                             par.estimators == solucao.estimators and
                             par.max_depth == solucao.max_depth and
                             par.booster == solucao.booster and
                             par.reg_lambda == solucao.reg_lambda and
                             par.reg_alpha == solucao.reg_alpha, self.solucoes))

        if search:
            solucao = search[0]

        else:
            updater = "coord_descent" if solucao.booster == "gblinear" else None
            modelo = XGBRegressor(random_state=self.seed,
                                  n_estimators=solucao.estimators,
                                  max_depth=solucao.max_depth,
                                  booster=solucao.booster,
                                  reg_lambda=solucao.reg_lambda,
                                  reg_alpha=solucao.reg_alpha,
                                  updater=updater)

            solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return solucao.fitness

## Particle Swarm Optimization (PSO)

### ESN

In [13]:
class PSOESN(Otimizador):
    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "PSO-ESN"
        self.run()

    def run(self):
        lower_bound = [10, 0.001, 0.001]
        uppper_bound = [400, 0.8, 0.8]
        bounds = (lower_bound, uppper_bound)

        options = {'c1': 0.5, 'c2': 0.5, 'w': 0.5}
        optimizer = GlobalBestPSO(n_particles=self.n_solucoes,
                                  dimensions=3,
                                  options=options,
                                  bounds=bounds)

        optimizer.optimize(self.get_fitness, iters=self.n_iteracoes)

    def get_fitness(self, parts):
        fit_lst = [self.run_objective_function(parts[j]) for j in range(self.n_solucoes)]

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return fit_lst

    def run_objective_function(self, particle_arr):
        solucao = SolucaoESN()
        solucao.n_reservoirs = int(particle_arr[0])
        solucao.sparsity = round(particle_arr[1], 4)
        solucao.spectral_radius = round(particle_arr[2], 4)

        search = list(filter(lambda par:
                             par.n_reservoirs == solucao.n_reservoirs and
                             par.sparsity == solucao.sparsity and
                             par.spectral_radius == solucao.spectral_radius, self.solucoes))

        if search:
            self.solucoes.append(search[0])
            return search[0].fitness

        modelo = ESN(n_inputs=self.dataset[self.features].shape[1],
                     n_outputs=1,
                     n_reservoir=solucao.n_reservoirs,
                     sparsity=solucao.sparsity,
                     spectral_radius=solucao.spectral_radius,
                     random_state=self.seed)

        solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        return solucao.fitness


### MLP

In [14]:
class PSOMLP(Otimizador):
    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "PSO-MLP"
        self.ACTIVATIONS = ["identity", "logistic", "tanh", "relu"]
        self.run()

    def run(self):
        lower_bound = [10, 0, 0]
        uppper_bound = [400, 1.0, 4]
        bounds = (lower_bound, uppper_bound)

        options = {'c1': 0.5, 'c2': 0.5, 'w': 0.5}
        optimizer = GlobalBestPSO(n_particles=self.n_solucoes,
                                  dimensions=3,
                                  options=options,
                                  bounds=bounds)

        optimizer.optimize(self.get_fitness, iters=self.n_iteracoes)

    def get_fitness(self, parts):
        fit_lst = [self.run_objective_function(parts[j]) for j in range(self.n_solucoes)]

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return fit_lst

    def run_objective_function(self, particle_arr):
        solucao = SolucaoMLP()
        solucao.hidden_layer_sizes = int(particle_arr[0])
        solucao.alpha = particle_arr[1]
        solucao.activation = self.ACTIVATIONS[int(particle_arr[2])]

        search = list(filter(lambda par:
                             par.hidden_layer_sizes == solucao.hidden_layer_sizes and
                             par.alpha == solucao.alpha and
                             par.activation == solucao.activation, self.solucoes))

        if search:
            self.solucoes.append(search[0])
            return search[0].fitness

        modelo = MLPRegressor(hidden_layer_sizes=(solucao.hidden_layer_sizes,),
                              activation=solucao.activation,
                              alpha=solucao.alpha,
                              random_state=self.seed)

        solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        return solucao.fitness

### Random Forest


In [15]:
class PSORF(Otimizador):
    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "PSO-RF"
        self.run()

    def run(self):
        lower_bound = [10, 10, 2, 2]
        uppper_bound = [400, 400, 50, 50]
        bounds = (lower_bound, uppper_bound)

        options = {'c1': 0.5, 'c2': 0.5, 'w': 0.5}
        optimizer = GlobalBestPSO(n_particles=self.n_solucoes,
                                  dimensions=4,
                                  options=options,
                                  bounds=bounds)

        optimizer.optimize(self.get_fitness, iters=self.n_iteracoes)

    def get_fitness(self, parts):
        fit_lst = [self.run_objective_function(parts[j]) for j in range(self.n_solucoes)]

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return fit_lst

    def run_objective_function(self, particle_arr):
        solucao = SolucaoRF()
        solucao.estimators = int(particle_arr[0])
        solucao.max_depth = int(particle_arr[1])
        solucao.min_samples_split = int(particle_arr[2])
        solucao.min_samples_leaf = int(particle_arr[3])

        search = list(filter(lambda par:
                             par.estimators == solucao.estimators and
                             par.max_depth == solucao.max_depth and
                             par.min_samples_split == solucao.min_samples_split and
                             par.min_samples_leaf == solucao.min_samples_leaf, self.solucoes))

        if search:
            self.solucoes.append(search[0])
            return search[0].fitness

        modelo = RandomForestRegressor(random_state=self.seed,
                                       n_estimators=solucao.estimators,
                                       max_depth=solucao.max_depth,
                                       min_samples_split=solucao.min_samples_split,
                                       min_samples_leaf=solucao.min_samples_leaf)

        solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        return solucao.fitness


### XGBoost

In [16]:
class PSOXGB(Otimizador):
    def __init__(self, dataset, features, n_solucoes, n_iteracoes, seed):
        super().__init__(dataset, features, n_solucoes, n_iteracoes, seed)
        self.nome_modelo = "PSO-XGBoost"
        self.BOOSTERS = ["gbtree", "gblinear", "dart"]
        self.run()

    def run(self):
        lower_bound = [10, 10, 0, 0, 0]
        uppper_bound = [400, 400, 2, 1, 1]
        bounds = (lower_bound, uppper_bound)

        options = {'c1': 0.5, 'c2': 0.5, 'w': 0.5}
        optimizer = GlobalBestPSO(n_particles=self.n_solucoes,
                                  dimensions=5,
                                  options=options,
                                  bounds=bounds)

        optimizer.optimize(self.get_fitness, iters=self.n_iteracoes)

    def get_fitness(self, parts):
        fit_lst = [self.run_objective_function(parts[j]) for j in range(self.n_solucoes)]

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv()

        return fit_lst

    def run_objective_function(self, particle_arr):
        solucao = SolucaoXGB()
        solucao.estimators = int(particle_arr[0])
        solucao.max_depth = int(particle_arr[1])
        solucao.booster = self.BOOSTERS[int(particle_arr[2])]
        solucao.reg_lambda = round(particle_arr[3], 4)
        solucao.reg_alpha = round(particle_arr[4], 4)

        search = list(filter(lambda par:
                             par.estimators == solucao.estimators and
                             par.max_depth == solucao.max_depth and
                             par.booster == solucao.booster and
                             par.reg_lambda == solucao.reg_lambda and
                             par.reg_alpha == solucao.reg_alpha, self.solucoes))

        if search:
            self.solucoes.append(search[0])
            return search[0].fitness

        updater = "coord_descent" if solucao.booster == "gblinear" else None
        modelo = XGBRegressor(random_state=self.seed,
                              n_estimators=solucao.estimators,
                              max_depth=solucao.max_depth,
                              booster=solucao.booster,
                              reg_lambda=solucao.reg_lambda,
                              reg_alpha=solucao.reg_alpha,
                              updater=updater)

        solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        return solucao.fitness


## Coyote Optimization Algorithm com Weaker Separator Booster (COA-WSB)

In [17]:
class COAWSB(Otimizador):
    def __init__(self, dataset, features, n_iteracoes, seed, tipo_treino, campus):
        super().__init__(dataset, features, n_iteracoes=n_iteracoes, seed=seed)
        self.nome_modelo = F"COA-WSB-{tipo_treino}-{campus}"
        self.tipo_treino = tipo_treino
        self.campus = campus
        self.PREVISORES = [m for m in MODELOS if m != "WSB-LOCAL" and m != "WSB-GLOBAL"]
        self.PESOS = np.arange(-1, 0, 0.0005)
        self.run()

    def run(self):
        lu = np.array([
            [0],
            [len(self.PESOS) - 1]
        ])
        COA(self.run_objective_function, lu, self.n_iteracoes)

    def objective_function(self, modelo):
        cvs_todos = []

        if self.tipo_treino == "LOCAL":
            dados = self.dataset[self.dataset["CAMPUS"] == self.campus].sort_values("DATA")
            treino, teste = train_test_split(dados, test_size=12, shuffle=False)
            x_treino = treino[self.features]
            y_treino = treino["CONSUMO"]
            x_teste = teste[self.features]
            y_teste = teste["CONSUMO"]

            previsoes = []
            historico = treino[["CONSUMO"]].copy()

            modelo.fit(x_treino.to_numpy(), y_treino.to_numpy())

            for i_test in range(12):
                row = x_teste.iloc[[i_test]].copy()
                historico = pd.concat([historico, pd.DataFrame([0], columns=["CONSUMO"], index=[i_test])], axis=0)

                # Recalcula os lags conforme os valores previstos pelo modelo
                lags = pd.DataFrame({f'LAG_{i:02d}': historico["CONSUMO"].shift(i) for i in range(1, 12 + 1) if
                                     f'LAG_{i:02d}' in self.features}).tail(1)
                row.update(lags)

                prev = modelo.predict(row.to_numpy(), i_test/12)[0]

                row["CONSUMO"] = prev
                previsoes.append(prev)
                historico.update(row)

            cvs_todos.append(calcular_rrmse(y_teste, previsoes).mean())

        elif self.tipo_treino == "GLOBAL":
            dados_treino = []
            dados_teste = {}

            for campus, dados in self.dataset.groupby("CAMPUS"):
                treino, teste = train_test_split(dados, test_size=12, shuffle=False)
                dados_treino.append(treino)
                dados_teste[campus] = teste

            dados_treino = pd.concat(dados_treino, ignore_index=True)

            x_treino = dados_treino[self.features]
            y_treino = dados_treino["CONSUMO"]
            x_teste = dados_teste[self.campus][self.features]
            y_teste = dados_teste[self.campus]["CONSUMO"]

            previsoes = []
            historico = dados_treino[dados_treino["CAMPUS"] == self.campus][["CONSUMO"]].copy()

            modelo.fit(x_treino.to_numpy(), y_treino.to_numpy())

            for i_test in range(12):
                row = x_teste.iloc[[i_test]].copy()
                historico = pd.concat([historico, pd.DataFrame([0], columns=["CONSUMO"], index=[i_test])], axis=0)

                # Recalcula os lags conforme os valores previstos pelo modelo
                lags = pd.DataFrame({f'LAG_{i:02d}': historico["CONSUMO"].shift(i) for i in range(1, 12 + 1) if
                                     f'LAG_{i:02d}' in self.features}).tail(1)
                row.update(lags)

                prev = modelo.predict(row.to_numpy(), i_test/12)[0]

                row["CONSUMO"] = prev
                previsoes.append(prev)
                historico.update(row)

            cvs_todos.append(calcular_rrmse(y_teste, previsoes).mean())

        else:
            raise ValueError("Tipo de treino inválido. Use 'LOCAL' ou 'GLOBAL'.")

        return np.array(cvs_todos).mean()

    def run_objective_function(self, parametros):
        solucao = SolucaoWSB()
        solucao.strong_predictor = "ESN" if self.tipo_treino == "GLOBAL" else "XGBoost"
        solucao.weight_g = self.PESOS[int(round(parametros[0]))]

        search = list(filter(lambda coyote:
                             coyote.strong_predictor == solucao.strong_predictor and
                             coyote.weight_g == solucao.weight_g, self.solucoes))

        if search:
            solucao = search[0]

        else:
            modelo = WSB(strong_predictor=get_modelo(solucao.strong_predictor),
                         weak_predictors=[get_modelo(p) for p in self.PREVISORES if p != solucao.strong_predictor],
                         weight_g=solucao.weight_g)

            solucao.fitness = self.objective_function(modelo)

        self.solucoes.append(solucao)

        self.solucoes = sorted(self.solucoes, key=lambda a: a.fitness)
        best = self.solucoes[0]
        self.iteracoes.append(best)
        self.salvar_csv(f"COA-WSB-{self.tipo_treino}/")

        return solucao.fitness

# Carregar Datasets

In [18]:
df_consumo = pd.read_csv("./dados/dados_normalizados_lagados.csv", sep=';', decimal='.')

dataframes = []
for campus, dados in df_consumo.groupby("CAMPUS"):
    if len(dados) < 100:
        continue
    treino, teste = train_test_split(dados, test_size=12, shuffle=False)
    dataframes.append(treino)

df_consumo = pd.concat(dataframes, ignore_index=True)

df_features = pd.read_csv("resultados/features/fitness_features_regressao.csv", sep=";", decimal=".")

df_features = df_features.sort_values("RRMSE").head(1).reset_index(drop=True)
df_features = pd.DataFrame(
    columns=str(df_features.iloc[0]["FEATURES"]).replace("(", '').replace(")", '').replace("'", "").split(", "))

df_consumo = df_consumo.sort_values("CAMPUS").sort_values("DATA")
df_features = df_features.columns
df_features

Index(['TEMP_MÉD_MIN_MENS', 'TEMP_MÉD_MÉD_MENS', 'PRECIPITAÇÃO_MÉD_MENS',
       'TEMP_MIN_MAX_MENS', 'TEMP_MAX_MIN_MENS', 'PRECIPITAÇÃO_MIN_MENS',
       'TEMP_MAX_MAX_MENS', 'DIA_DA_SEMANA_dom', 'DIA_DA_SEMANA_seg',
       'DIA_DA_SEMANA_sáb', 'DIA_DA_SEMANA_ter', 'MÊS_abr', 'MÊS_ago',
       'MÊS_fev', 'MÊS_jun', 'MÊS_mai', 'MÊS_nov', 'ANO_2021', 'ANO_2022',
       'ANO_2023', 'ANO_2015', 'ANO_2016', 'ANO_2017', 'ANO_2018', 'ANO_2019',
       'CAMPUS_ASTORGA', 'CAMPUS_CAMPO LARGO', 'CAMPUS_CAPANEMA',
       'CAMPUS_CASCAVEL', 'CAMPUS_CORONEL VIVIDA', 'CAMPUS_CURITIBA',
       'CAMPUS_GOIOERÊ', 'CAMPUS_IVAIPORÃ', 'CAMPUS_JAGUARIAÍVA',
       'CAMPUS_LONDRINA - CENTRO', 'CAMPUS_PALMAS', 'CAMPUS_PARANAGUÁ',
       'CAMPUS_PINHAIS', 'CAMPUS_TELÊMACO BORBA', 'CAMPUS_UMUARAMA',
       'CURSOS_TEC_SUBSEQUENTE', 'CURSOS_GRAD_MATUTINO',
       'CURSOS_GRAD_VESPERTINO', 'CURSOS_GRAD_NOTURNO', 'CURSOS_POS', 'FÉRIAS',
       'COVID', 'LAG_01', 'LAG_02', 'LAG_03', 'LAG_05', 'LAG_07', 'LAG_09'],


# Execução da Otimização (ESN, MLP, RF e XGBoost)

In [19]:
# for seed in SEEDS:
# PSOESN(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)
# PSOMLP(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)
# PSORF(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)
# PSOXGB(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)
# SAESN(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)
# SAMLP(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)
# SARF(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)
# SAXGB(df_consumo, df_features, N_SOLUCOES, N_ITER, seed)


# Execução da Otimização (COA-WSB)

## Melhores Parâmetros dos outros modelos

In [4]:
def get_modelo(nome):
    if nome == "ESN":
        return ESN(n_inputs=df_consumo[df_features].shape[1],
                   n_outputs=1,
                   n_reservoir=int(best["ESN"]["Reservoirs"]),
                   sparsity=best["ESN"]["Sparsity"],
                   spectral_radius=best["ESN"]["Spectral Radius"],
                   random_state=int(best["ESN"]["SEED"]))

    if nome == "MLP":
        mlp = MLPRegressor(hidden_layer_sizes=(int(best["MLP"]["Hidden Layers"]),),
                           activation=best["MLP"]["Activation"],
                           alpha=best["MLP"]["Alpha"],
                           random_state=int(best["MLP"]["SEED"]))
        return mlp

    if nome == "RF":
        return RandomForestRegressor(random_state=int(best["RF"]["SEED"]),
                                     n_estimators=int(best["RF"]["N_estimators"]),
                                     max_depth=int(best["RF"]["Max_depth"]),
                                     min_samples_split=int(best["RF"]["Min_samples_split"]),
                                     min_samples_leaf=int(best["RF"]["Min_samples_leaf"]))

    if nome == "XGBoost":
        return XGBRegressor(random_state=int(best["XGBoost"]["SEED"]),
                            n_estimators=int(best["XGBoost"]["N_estimators"]),
                            max_depth=int(best["XGBoost"]["Max_depth"]),
                            booster=best["XGBoost"]["Booster"],
                            reg_lambda=best["XGBoost"]["Lambda"],
                            reg_alpha=best["XGBoost"]["Alpha"],
                            updater="coord_descent" if best["XGBoost"]["Booster"] == "gblinear" else None)


best = {}
for modelo in MODELOS:
    df_aux = pd.read_csv(
        f"./resultados/otimização - regressão/BEST-{modelo}.csv", sep=';',
        decimal='.', header=0)
    best[modelo] = df_aux.iloc[0]
best

{'ESN': OTIMIZADOR           PSO
 MODELO               ESN
 SEED                1000
 Reservoirs         384.0
 Sparsity            0.43
 Spectral Radius     0.75
 Fitness            0.302
 Name: 0, dtype: object,
 'MLP': OTIMIZADOR         PSO
 MODELO             MLP
 SEED              2000
 Hidden Layers      296
 Alpha            0.947
 Activation        tanh
 Fitness          0.238
 Name: 0, dtype: object,
 'RF': OTIMIZADOR              SA
 MODELO                  RF
 SEED                  2000
 N_estimators         218.0
 Max_depth            103.0
 Min_samples_split      9.0
 Min_samples_leaf       2.0
 Fitness               0.26
 Name: 0, dtype: object,
 'XGBoost': OTIMIZADOR          PSO
 MODELO          XGBoost
 SEED               2000
 N_estimators        385
 Max_depth           119
 Booster          gbtree
 Lambda            0.765
 Alpha             0.058
 Fitness           0.245
 Name: 0, dtype: object}

## Seleção dos Parâmetros do WSB

In [ ]:
df_consumo = pd.read_csv("./dados/dados_normalizados_lagados.csv", sep=';', decimal='.')

dataframes = []
for campus, dados in df_consumo.groupby("CAMPUS"):
    treino, teste = train_test_split(dados, test_size=12, shuffle=False)
    dataframes.append(treino)
dataframes = pd.concat(dataframes, ignore_index=True)

for campus in dataframes["CAMPUS"].unique():
    for seed in SEEDS:
        COAWSB(dataframes, df_features, 10, seed, "LOCAL", campus)
        COAWSB(dataframes, df_features, 10, seed, "GLOBAL", campus)


# Resultados da Otimização
## Evolução da FO - PSO E SA

In [11]:
def formatar_y(valor, pos):
    if valor >= 1:
        return f"{valor:.2f}+"
    return f"{valor:.2f}"


for modelo in MODELOS:
    plt.figure(figsize=(6, 4))
    plt.rcParams['xtick.labelsize'] = 18
    plt.rcParams['ytick.labelsize'] = 18
    plt.rcParams.update({'font.size': 18})
    plt.rcParams['axes.prop_cycle'] = plt.cycler(
        color=["blue", colors.red_rgb])
    df_melhor = []

    for otimizador in OTIMIZADORES:
        df = []
        for seed in SEEDS:
            try:
                novo_df = pd.read_csv(f'resultados/otimização - regressão/{otimizador}-{modelo} SEED {seed}.csv',
                                      sep=";", decimal=".", header=0)
                novo_df["SEED"] = seed
                novo_df["MODELO"] = modelo
                novo_df["OTIMIZADOR"] = otimizador
                novo_df = novo_df[["OTIMIZADOR", "MODELO", "SEED"] + [col for col in novo_df.columns if
                                                                      col not in ["OTIMIZADOR", "MODELO",
                                                                                  "SEED"]]].drop(
                    columns=["Unnamed: 0"])

            except Exception as e:
                print(e)
                continue

            if otimizador == "PSO":
                novo_df = pd.concat([novo_df] * N_ITER, ignore_index=True)
                novo_df = novo_df.sort_values(by=["Fitness"], ascending=False)
            df.append(novo_df)

        df = pd.concat(df, ignore_index=True)
        df = np.round(df, decimals=3)

        df_best = df.sort_values(by=["Fitness"], ascending=False).tail(1)
        df_best = df[df["SEED"] == df_best["SEED"].iloc[0]]
        df_best.to_csv(
            f"./resultados/otimização - regressão/BEST SEED {otimizador}-{modelo}.csv",
            sep=';',
            decimal='.')
        best_seed = df_best["SEED"].iloc[0]
        best_fitness = df_best["Fitness"].tail(1).iloc[0]

        df_melhor.append(df_best.tail(1))
        plt.plot(range(1, len(df_best) + 1), [val["Fitness"] for index, val in df_best.iterrows()],
                 label=f"{otimizador}-{modelo}")

        color = plt.rcParams['axes.prop_cycle'].by_key()['color'][OTIMIZADORES.index(otimizador)]
        plt.annotate(f"{best_fitness:.3f}",
                     xy=(N_ITER*N_SOLUCOES * (0.89 if otimizador == "PSO" else 0.76), best_fitness + 0.005),
                     fontsize=12,
                     color=color)
    plt.xlabel('Avaliações da FO')
    plt.ylabel('RRMSE')
    plt.gca().yaxis.set_major_formatter(FuncFormatter(formatar_y))
    plt.ylim(0.2, 0.5)
    plt.xlim(0, N_ITER * N_SOLUCOES)
    ax = plt.gca()
    ax.set_facecolor('white')
    plt.grid(True, color='grey', linestyle="--", linewidth=0.5)
    plt.legend(facecolor='white')
    plt.savefig(f"./resultados/otimização - regressão/{modelo}.png", bbox_inches='tight')
    plt.close()

    df_melhor = pd.concat(df_melhor, ignore_index=True)
    df_melhor = df_melhor.sort_values(by=["Fitness"], ascending=False).tail(1)
    df_melhor.to_csv(f"./resultados/otimização - regressão/BEST-{modelo}.csv",
                     sep=';',
                     decimal='.', index=False)


## Evolução da FO - COA

In [8]:
df_consumo = pd.read_csv("./dados/dados_normalizados_lagados.csv", sep=';', decimal='.')

for campus in df_consumo["CAMPUS"].unique():
    plt.figure(figsize=(6, 4))
    plt.rcParams['xtick.labelsize'] = 18
    plt.rcParams['ytick.labelsize'] = 18
    plt.rcParams.update({'font.size': 18})
    plt.rcParams['axes.prop_cycle'] = plt.cycler(
        color=["blue", colors.red_rgb])

    for modelo in MODELOS_WSB:
        df = []
        for seed in SEEDS:
            try:
                novo_df = pd.read_csv(f'resultados/otimização - regressão/COA-{modelo}/COA-{modelo}-{campus} SEED {seed}.csv',
                                      sep=";", decimal=".", header=0)
                novo_df["SEED"] = seed
                novo_df["MODELO"] = f"{modelo}-{campus}"
                novo_df["OTIMIZADOR"] = "COA"
                novo_df = novo_df[["OTIMIZADOR", "MODELO", "SEED"] + [col for col in novo_df.columns if
                                                                      col not in ["OTIMIZADOR", "MODELO",
                                                                                  "SEED"]]].drop(
                    columns=["Unnamed: 0"])
                df.append(novo_df)
            except Exception as e:
                print(e)
                continue

        df = pd.concat(df, ignore_index=True)
        df = np.round(df, decimals=3)

        df_best = df.sort_values(by=["Fitness"], ascending=False).tail(1)
        df_best = df[df["SEED"] == df_best["SEED"].iloc[0]]
        df_best.to_csv(
            f"./resultados/otimização - regressão/COA-{modelo}/BEST SEED COA-{modelo}-{campus}.csv",
            sep=';',
            decimal='.')
        best_seed = df_best["SEED"].iloc[0]
        best_fitness = df_best["Fitness"].tail(1).iloc[0]

        df_best.tail(1).to_csv(f"./resultados/otimização - regressão/COA-{modelo}/BEST-{modelo}-{campus}.csv",
                               sep=';',
                               decimal='.', index=False)
        plt.plot(range(1, len(df_best) + 1), df_best["Fitness"] ,
                 label=f"COA-{modelo}")

        color = plt.rcParams['axes.prop_cycle'].by_key()['color'][MODELOS_WSB.index(modelo)]
        plt.annotate(f"{best_fitness:.3f}",
                     xy=(len(df_best) * 0.89, best_fitness + 0.005),
                     fontsize=12,
                     color=color)

    plt.title(campus)
    plt.xlabel('Avaliações da FO')
    plt.ylabel('RRMSE')
    plt.gca().yaxis.set_major_formatter(FuncFormatter(formatar_y))
    plt.ylim(0.2, 0.5)
    plt.xlim(0, 100)
    ax = plt.gca()
    ax.set_facecolor('white')
    plt.grid(True, color='grey', linestyle="--", linewidth=0.5)
    plt.legend(facecolor='white')
    plt.savefig(f"./resultados/otimização - regressão/WSB-{campus}.png", bbox_inches='tight')
    plt.close()


## Melhores Hiperparâmetros

In [10]:
for modelo in MODELOS:
    best = {}
    df_aux = pd.read_csv(
        f"./resultados/otimização - regressão/BEST-{modelo}.csv", sep=';',
        decimal='.', header=0)

    best[modelo] = pd.concat([best[modelo] if modelo in best.keys() else pd.DataFrame(),
                              pd.DataFrame(df_aux)])
    display(df_aux)

best = []
df_consumo = pd.read_csv("./dados/dados_normalizados_lagados.csv", sep=';', decimal='.')
for modelo in MODELOS_WSB:
    for campus in df_consumo["CAMPUS"].unique():
        df_aux = pd.read_csv(
            f"./resultados/otimização - regressão/COA-{modelo}/BEST-{modelo}-{campus}.csv", sep=';',
            decimal='.', header=0)
        best.append(df_aux)
best = pd.concat(best, ignore_index=True)
display(best)


,OTIMIZADOR,MODELO,SEED,Reservoirs,Sparsity,Spectral Radius,Fitness
0,PSO,ESN,9000,11.0,0.23,0.639,0.289


,OTIMIZADOR,MODELO,SEED,Hidden Layers,Alpha,Activation,Fitness
0,PSO,MLP,10000,220,0.979,relu,0.227


,OTIMIZADOR,MODELO,SEED,N_estimators,Max_depth,Min_samples_split,Min_samples_leaf,Fitness
0,PSO,RF,7000,15.0,232.0,11.0,4.0,0.247


,OTIMIZADOR,MODELO,SEED,N_estimators,Max_depth,Booster,Lambda,Alpha,Fitness
0,PSO,XGBoost,5000,242,119,gbtree,0.898,0.041,0.243


,OTIMIZADOR,MODELO,SEED,Strong_predictor,Weight_g,Fitness
0,COA,WSB-GLOBAL-ASSIS CHATEAUBRIAND,10000,ESN,-0.003,0.289
1,COA,WSB-GLOBAL-ASTORGA,6000,ESN,-1.000,0.287
2,COA,WSB-GLOBAL-BARRACÃO,10000,ESN,-0.989,0.297
3,COA,WSB-GLOBAL-CAMPO LARGO,10000,ESN,-0.989,0.321
4,COA,WSB-GLOBAL-CAPANEMA,10000,ESN,-0.003,0.255
5,COA,WSB-GLOBAL-CASCAVEL,1000,ESN,-0.990,0.270
6,COA,WSB-GLOBAL-CORONEL VIVIDA,1000,ESN,-0.990,0.234
7,COA,WSB-GLOBAL-CURITIBA,1000,ESN,-0.797,0.684
8,COA,WSB-GLOBAL-FOZ DO IGUAÇU,1000,ESN,-0.990,0.316
9,COA,WSB-GLOBAL-GOIOERÊ,1000,ESN,-0.531,0.275
